# Sentence Similarity — Hybrid Embedding (LSA + BM25)

In [2]:
import sys
import numpy as np
sys.path.insert(0, '.')
from main import HybridEmbedder

model = HybridEmbedder.load('msmarco_data/msmarco_model')
print(model.lsa)
print(model.bm25)

[LSAEncoder] loaded from msmarco_data/msmarco_model_lsa
[BM25Encoder] loaded from msmarco_data/msmarco_model_bm25
LSAEncoder(n_components=128, vocab_size=48850, k=128)
BM25Encoder(k1=1.5, b=0.75, vocab_size=48850, avgdl=32.4)


In [ ]:
sentence_A = "the dog is sleeping on the floor"
sentence_B = "the dog is awake and running around"
alpha = 0.5   # 0 = pure dense (semantic)  |  1 = pure sparse (keyword)

dense_A, sparse_vec_A = model.encode(sentence_A)   # dense: (128,)  sparse: (V,)
dense_B, sparse_vec_B = model.encode(sentence_B)

#  1. Dense cosine (LSA) 
dense_cosine = float(np.dot(dense_A, dense_B))

#  2. Sparse cosine (BM25) 
norm_A = np.linalg.norm(sparse_vec_A)
norm_B = np.linalg.norm(sparse_vec_B)
if norm_A > 1e-12 and norm_B > 1e-12:
    sparse_cosine = float(np.dot(sparse_vec_A, sparse_vec_B) / (norm_A * norm_B))
else:
    sparse_cosine = 0.0

#  3. Hybrid score 
hybrid = (1 - alpha) * dense_cosine + alpha * sparse_cosine

print(f"A : {sentence_A!r}")
print(f"B : {sentence_B!r}")
print()
print(f"  Dense  cosine  (LSA)          : {dense_cosine:.4f}")
print(f"  Sparse cosine  (BM25)         : {sparse_cosine:.4f}")
print(f"  Hybrid (alpha={alpha})        : {hybrid:.4f}   <-- final score")
print()
print("  Score guide: -1=opposite  0=unrelated  1=identical")

sparse_A = model.bm25.encode_as_dict(sentence_A)
sparse_B = model.bm25.encode_as_dict(sentence_B)
rev = {v: k for k, v in model.bm25.vocabulary.items()}
shared = sorted([(rev[t], sparse_A[t], sparse_B[t]) for t in sparse_A if t in sparse_B],
                key=lambda x: -(x[1] + x[2]))
print()
if shared:
    print(f"  Shared keywords ({len(shared)}):")
    for term, wa, wb in shared:
        print(f"    '{term}'   A={wa:.2f}  B={wb:.2f}")
else:
    print("  No shared keywords — sparse cosine = 0, hybrid relies on dense only")

A : 'the dog is sleeping on the floor'
B : 'the dog is awake and running around'

  Dense  cosine  (LSA)          : 0.4463
  Sparse cosine  (BM25)         : 0.2375
  Hybrid (alpha=0.5)        : 0.3419   <-- final score

  Score guide: -1=opposite  0=unrelated  1=identical

  Shared keywords (1):
    'dog'   A=9.03  B=8.83


In [ ]:
import csv, time
import numpy as np

#  Load STS-B test pairs 
pairs = []
with open('stsb_test.csv', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        pairs.append((row['sentence1'], row['sentence2'], float(row['score'])))

print(f"Loaded {len(pairs)} STS-B pairs")

#  Score every pair 
gold, dense_scores, sparse_scores = [], [], []

t0 = time.time()
for s1, s2, label in pairs:
    d1, sp1 = model.encode(s1)
    d2, sp2 = model.encode(s2)

    dc = float(np.dot(d1, d2))

    n1, n2 = np.linalg.norm(sp1), np.linalg.norm(sp2)
    sc = float(np.dot(sp1, sp2) / (n1 * n2)) if n1 > 1e-12 and n2 > 1e-12 else 0.0

    # remap gold [0, 1]  ->  [-1, 1]  so all scores share the same gold scale
    gold.append(2 * label - 1)
    dense_scores.append(dc)
    sparse_scores.append(sc)

print(f"Encoded {len(pairs)} pairs in {time.time()-t0:.1f}s")
print()
print("Score scale:  -1 = completely opposite  |  0 = unrelated  |  1 = identical")

gold   = np.array(gold)
dense  = np.array(dense_scores)
sparse = np.array(sparse_scores)

def pearson(x, y):
    x, y = x - x.mean(), y - y.mean()
    denom = np.linalg.norm(x) * np.linalg.norm(y)
    return float(np.dot(x, y) / denom) if denom > 1e-12 else 0.0

def spearman(x, y):
    rx = np.argsort(np.argsort(x)).astype(float)
    ry = np.argsort(np.argsort(y)).astype(float)
    return pearson(rx, ry)

#  Evaluate across alpha blends 
print()
print(f"{'Mode':<22}  {'Pearson':>8}  {'Spearman':>9}")
print("-" * 44)

best_p, best_alpha = -1, 0.5
for alpha in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    hybrid = (1 - alpha) * dense + alpha * sparse
    p = pearson(hybrid, gold)
    s = spearman(hybrid, gold)
    name = "dense only" if alpha == 0 else "sparse only" if alpha == 1 else f"hybrid a={alpha}"
    print(f"  {name:<20}  {p:>8.4f}  {s:>9.4f}")
    if p > best_p:
        best_p, best_alpha = p, alpha

best_hybrid = (1 - best_alpha) * dense + best_alpha * sparse
print()
print(f"  Best alpha = {best_alpha}  ->  Pearson {best_p:.4f}  Spearman {spearman(best_hybrid, gold):.4f}")
print()


Loaded 19656 STS-B pairs
Encoded 19656 pairs in 164.4s

Score scale:  -1 = completely opposite  |  0 = unrelated  |  1 = identical

Mode                     Pearson   Spearman
--------------------------------------------
  dense only              0.3913     0.3930
  hybrid a=0.1            0.4322     0.4353
  hybrid a=0.2            0.4761     0.4786
  hybrid a=0.3            0.5220     0.5224
  hybrid a=0.4            0.5680     0.5666
  hybrid a=0.5            0.6117     0.6103
  hybrid a=0.6            0.6497     0.6512
  hybrid a=0.7            0.6787     0.6846
  hybrid a=0.8            0.6964     0.7051
  hybrid a=0.9            0.7018     0.7114
  sparse only             0.6961     0.7022

  Best alpha = 0.9  ->  Pearson 0.7018  Spearman 0.7114

Sample predictions  (gold scale: -1..1)
------------------------------------------------------------------------
  gold=-0.267  pred=+0.223  | This church choir sings to the masses as they...
                         | The church has cra